# x402 `exact` scheme — seller-side facilitator test

Tests the **`exact`** scheme's full seller-facing flow directly against a running
facilitator: discover it via `/supported`, approve its USDC fee allowance, then walk
one payment through `/verify` → `/settle` and confirm the fee was actually collected.

**Where this fits among the sibling notebooks:**
- **This notebook** — `exact` scheme, seller's perspective (what you'd build if you're
  integrating a server against this facilitator).
- [`x402_batch_settlement.ipynb`](./x402_batch_settlement.ipynb) — same idea, for the
  `batch-settlement` scheme.
- [`genimg_x402_buyer.ipynb`](./genimg_x402_buyer.ipynb) — the buyer's perspective
  instead: a real production integration (`scw_js`'s genimg endpoint) paid via
  `@x402/fetch`, with no direct facilitator calls at all.

## Fee flow

1. **Buyer** signs an EIP-3009 authorization: `transferWithAuthorization(buyer → seller, amount)`.
2. **Facilitator** verifies the signature and checks the seller's USDC allowance for the fee.
3. **Facilitator** settles: executes the `transferWithAuthorization` on-chain (buyer → seller).
4. **Facilitator** collects its fee: `transferFrom(seller → facilitator, feeAmount)`, post-settlement.

**Key point:** the **seller** pays the fee, not the buyer. The seller must pre-approve USDC
spending by the facilitator wallet — Step 3 below does that. The recommended approval is
deliberately small (1 USDC ≈ 100 settlements), since the facilitator's spender is a hot
wallet — re-approval is expected, not a one-time setup. Watch `remainingSettlements` in
the `/verify` response to know when.

## Prerequisites

1. Start the local facilitator: `cd x402_facilitator && npm run dev`
2. Set env vars in the package's single `x402_facilitator/.env` (one level up):
   - `TEST_WALLET_PRIVATE_KEY` — buyer's private key (needs testnet USDC)
   - `NFT_WALLET_PUBLIC_KEY` — seller's address (receives payment)
   - `NFT_WALLET_PRIVATE_KEY` — seller's private key (signs the fee approval)

In [ ]:
// Setup: Imports and Configuration
import { load } from "https://deno.land/std@0.224.0/dotenv/mod.ts";
import { privateKeyToAccount } from "npm:viem@2/accounts";
import { createPublicClient, createWalletClient, http, formatUnits, getContract } from "npm:viem@2";
import { optimism, optimismSepolia, base, baseSepolia } from "npm:viem@2/chains";

// Load environment variables
// Load from the package's single .env, one level up (x402_facilitator/.env).
// examplePath: null skips the "every key in ./.env.example must be present" check.
const env = await load({ envPath: "../.env", examplePath: null, export: true });

const BUYER_PRIVATE_KEY = env.TEST_WALLET_PRIVATE_KEY;
const SELLER_ADDRESS = env.NFT_WALLET_PUBLIC_KEY as `0x${string}`;
const SELLER_PRIVATE_KEY = env.NFT_WALLET_PRIVATE_KEY;

// Create buyer account
// Idempotent whether or not the .env value already carries the 0x prefix (it does, per
// .env.example) — a raw template literal here would double it and fail viem's hex parse.
const buyerAccount = privateKeyToAccount(`0x${BUYER_PRIVATE_KEY.replace(/^0x/, "")}`);
const sellerAccount = privateKeyToAccount(`0x${SELLER_PRIVATE_KEY.replace(/^0x/, "")}`);

console.log("🚀 x402 exact scheme — facilitator test");
console.log(`   Buyer:  ${buyerAccount.address}`);
console.log(`   Seller: ${SELLER_ADDRESS}`);
console.log(`   Seller (from key): ${sellerAccount.address}`);

## Network & Payment Configuration

**Supported Networks:**
| Chain | Testnet | Mainnet |
|-------|---------|---------|
| Optimism | `eip155:11155420` | `eip155:10` |
| Base | `eip155:84532` | `eip155:8453` |

**To switch networks:**
1. Set `USE_BASE = true` for Base, `false` for Optimism
2. Set `USE_MAINNET = true` for mainnet (⚠️ REAL MONEY!)
3. Set `TOKEN` to `"EURC"` or `"USDC"`. EURC exists on Base only, so it needs `USE_BASE = true`.
   The facilitator charges its fee in the token you pay in, so the seller approves that token.

The buyer pays `PAYMENT_AMOUNT` to the seller. The facilitator fee (0.01 of the paid token) is collected **from the seller** after settlement — the buyer pays nothing extra.

In [ ]:
// ═══════════════════════════════════════════════════════════════
// ⚠️ NETWORK SELECTION - Choose your network here
// ═══════════════════════════════════════════════════════════════

// Step 1: Choose the chain (Optimism or Base)
const USE_BASE = true;     // Set to true for Base, false for Optimism (Optimism has no EURC)

// Step 2: Choose testnet or mainnet
const USE_MAINNET = false; // Set to true for mainnet with REAL MONEY

// Step 3: Choose the token. EURC exists on Base only (Circle has no Optimism deployment).
const TOKEN: "USDC" | "EURC" = "EURC";

// Network configuration (all 4 combinations)
interface NetworkConfig {
    chain: typeof optimism;
    chainId: number;
    caip2Network: string;
    networkName: string;
    usdcAddress: `0x${string}`;
    usdcName: string;
}

const NETWORK_CONFIGS: Record<string, NetworkConfig> = {
    "optimism:testnet": {
        chain: optimismSepolia,
        chainId: 11155420,
        caip2Network: "eip155:11155420",
        networkName: "Optimism Sepolia (Testnet)",
        usdcAddress: "0x5fd84259d66Cd46123540766Be93DFE6D43130D7",
        usdcName: "USDC",
    },
    "optimism:mainnet": {
        chain: optimism,
        chainId: 10,
        caip2Network: "eip155:10",
        networkName: "Optimism Mainnet",
        usdcAddress: "0x0b2C639c533813f4Aa9D7837CAf62653d097Ff85",
        usdcName: "USD Coin",
    },
    "base:testnet": {
        chain: baseSepolia,
        chainId: 84532,
        caip2Network: "eip155:84532",
        networkName: "Base Sepolia (Testnet)",
        usdcAddress: "0x036CbD53842c5426634e7929541eC2318f3dCF7e",
        usdcName: "USDC",
    },
    "base:mainnet": {
        chain: base,
        chainId: 8453,
        caip2Network: "eip155:8453",
        networkName: "Base Mainnet",
        usdcAddress: "0x833589fCD6eDb6E08f4c7C32D4f71b54bdA02913",
        usdcName: "USD Coin",
    },
};

// Select config based on toggles
const configKey = `${USE_BASE ? "base" : "optimism"}:${USE_MAINNET ? "mainnet" : "testnet"}`;
const config = NETWORK_CONFIGS[configKey];

// Convenience aliases used in all downstream cells
const chainConfig = config.chain;

// EURC per network — address and EIP-712 domain name read on-chain (name() = "EURC", version "2").
const EURC_BY_NETWORK: Record<string, { address: `0x${string}`; name: string }> = {
    "eip155:84532": { address: "0x808456652fdb597867f38412077A9182bf77359F", name: "EURC" }, // Base Sepolia
    "eip155:8453": { address: "0x60a3E35Cc302bFA44Cb288Bc5a4F316Fdb1adb42", name: "EURC" },  // Base
};
const eurc = EURC_BY_NETWORK[config.caip2Network];
if (TOKEN === "EURC" && !eurc) {
    throw new Error(`EURC is not deployed on ${config.networkName} — set USE_BASE = true, or TOKEN = "USDC".`);
}
// The token the buyer pays in, and so the one the facilitator charges its fee in.
const tokenAddress = TOKEN === "EURC" ? eurc!.address : config.usdcAddress;
const tokenName = TOKEN === "EURC" ? eurc!.name : config.usdcName;

// What the buyer pays (seller receives this minus nothing — fee is separate!)
const PAYMENT_AMOUNT = "20000";  // 0.02 of the token (USDC and EURC both have 6 decimals)

if (USE_MAINNET) {
    console.log(`\n🚨 WARNING: Using REAL MONEY on ${config.networkName}!`);
} else {
    console.log(`\n🧪 Using testnet: ${config.networkName}`);
}
console.log(`   Chain: ${USE_BASE ? "Base" : "Optimism"}`);
console.log(`   Chain ID: ${config.chainId}`);
console.log(`   CAIP-2: ${config.caip2Network}`);
console.log(`   Token: ${tokenAddress}`);
console.log(`   Token Name: ${tokenName}`);
console.log(`   Payment: ${PAYMENT_AMOUNT} (${Number(PAYMENT_AMOUNT) / 1e6} ${TOKEN})`);

## Step 1: Query /supported

Check what the facilitator supports. The `facilitator_fee` extension shows the fee amount and the facilitator's wallet address (needed for USDC approval).

In [ ]:
// Facilitator endpoint (local dev server)
const FACILITATOR_URL = "http://localhost:8080";

const SUPPORTED_URL = `${FACILITATOR_URL}/supported`;
const VERIFY_URL = `${FACILITATOR_URL}/verify`;
const SETTLE_URL = `${FACILITATOR_URL}/settle`;

// Query /supported
const supportedResponse = await fetch(SUPPORTED_URL);
const supported = await supportedResponse.json();

console.log(`📡 GET ${SUPPORTED_URL} → ${supportedResponse.status}`);

// Show supported networks
console.log(`\n🌐 Supported Networks:`);
for (const kind of supported.kinds || []) {
    console.log(`   - ${kind.network} (scheme: ${kind.scheme}, x402v${kind.x402Version})`);
}

// Fee disclosure: `extensions` is only a list of key names; the detail lives in the
// top-level `facilitatorFees` object (x402 fee disclosure proposal, coinbase/x402#1016).
const fees = supported.facilitatorFees;
let facilitatorAddress: string | null = null;
let feeAmount: string | null = null;

if (fees) {
    facilitatorAddress = fees.recipient;
    feeAmount = fees.flatFee;
    console.log(`\n💸 Facilitator fee:`);
    console.log(`   Fee: ${fees.fee?.description}`);
    console.log(`   Amount: ${feeAmount} (${Number(feeAmount) / 1e6} ${TOKEN}, charged in the settled token)`);
    console.log(`   Facilitator Address: ${facilitatorAddress}`);
    console.log(`   Collection: ${fees.fee?.collection}`);
    console.log(`\n🔧 Setup Required:`);
    console.log(`   ${fees.setup?.description}`);
    console.log(`   Function: ${fees.setup?.function}`);
    console.log(`   Spender: ${fees.setup?.spender}`);
} else {
    console.log(`\n⚠️ No facilitatorFees disclosure found — fees may be disabled`);
}

// Show signers
if (supported.signers?.["eip155:*"]) {
    console.log(`\n📝 Facilitator Signers: ${supported.signers["eip155:*"].join(", ")}`);
}

> **Note:** the printed output above is captured from a past run, before Phase 5.2 reworded the `/supported` disclosure. A fresh run now shows a *recurring*-approval message recommending **1 USDC (100 settlements)**, not the "one-time" / "100 USDC (10,000 settlements)" text shown here — see `x402_facilitator/FEE_MODEL_PLAN.md` Phase 5.

## Step 2: Check USDC Balances (Before)

We check the USDC balances of all three parties before any transaction:
- **Buyer** — will pay the full `PAYMENT_AMOUNT`
- **Seller** — will receive `PAYMENT_AMOUNT`, then the facilitator takes the fee via `transferFrom`
- **Facilitator** — collects the fee after settlement

In [ ]:
// Minimal ERC-20 ABI for balance, allowance, and approval
const ERC20_ABI = [
    {
        name: "balanceOf",
        type: "function",
        stateMutability: "view",
        inputs: [{ name: "account", type: "address" }],
        outputs: [{ name: "", type: "uint256" }],
    },
    {
        name: "allowance",
        type: "function",
        stateMutability: "view",
        inputs: [
            { name: "owner", type: "address" },
            { name: "spender", type: "address" },
        ],
        outputs: [{ name: "", type: "uint256" }],
    },
    {
        name: "approve",
        type: "function",
        stateMutability: "nonpayable",
        inputs: [
            { name: "spender", type: "address" },
            { name: "value", type: "uint256" },
        ],
        outputs: [{ name: "", type: "bool" }],
    },
] as const;

// Create public client for reading on-chain state
const publicClient = createPublicClient({
    chain: chainConfig,
    transport: http(chainConfig.rpcUrls.default.http[0]),
});

// Helper to read token balance
async function getTokenBalance(address: `0x${string}`): Promise<bigint> {
    return publicClient.readContract({
        address: tokenAddress as `0x${string}`,
        abi: ERC20_ABI,
        functionName: "balanceOf",
        args: [address],
    });
}

// Check all balances
const balancesBefore = {
    buyer: await getTokenBalance(buyerAccount.address),
    seller: await getTokenBalance(SELLER_ADDRESS as `0x${string}`),
    facilitator: facilitatorAddress
        ? await getTokenBalance(facilitatorAddress as `0x${string}`)
        : 0n,
};

console.log(`💰 ${TOKEN} Balances Before:`);
console.log(`   Buyer     (${buyerAccount.address}): ${Number(balancesBefore.buyer) / 1e6} ${TOKEN}`);
console.log(`   Seller    (${SELLER_ADDRESS}): ${Number(balancesBefore.seller) / 1e6} ${TOKEN}`);
if (facilitatorAddress) {
    console.log(`   Facilitator (${facilitatorAddress}): ${Number(balancesBefore.facilitator) / 1e6} ${TOKEN}`);
}

// Check if buyer has enough
const paymentBigInt = BigInt(PAYMENT_AMOUNT);
if (balancesBefore.buyer < paymentBigInt) {
    console.log(`\n⚠️ Buyer needs at least ${Number(paymentBigInt) / 1e6} ${TOKEN} but only has ${Number(balancesBefore.buyer) / 1e6}`);
} else {
    console.log(`\n✅ Buyer has enough ${TOKEN} for payment`);
}

## Step 3: Seller Approves USDC for Fee Collection

The fee model requires the **seller** to have an active ERC-20 `approve` for the facilitator's address.
This is a **recurring approval** — the recommended amount is deliberately small (the facilitator's
spender is a hot wallet), so re-approve as `remainingSettlements` (from `/verify`) runs low.

**Flow:**
1. Seller calls `USDC.approve(facilitatorAddress, amount)` — e.g. 1 USDC = 100 settlements at 0.01 USDC fee
2. The facilitator checks `allowance(seller, facilitator)` during `/verify`
3. After `/settle`, the facilitator calls `transferFrom(seller, facilitator, fee)` to collect

If the allowance is already sufficient, you can skip the approval transaction.

In [ ]:
// First check existing allowance
const sellerAddr = SELLER_ADDRESS as `0x${string}`;
const facilitatorAddr = facilitatorAddress as `0x${string}`;

const currentAllowance = await publicClient.readContract({
    address: tokenAddress as `0x${string}`,
    abi: ERC20_ABI,
    functionName: "allowance",
    args: [sellerAddr, facilitatorAddr],
});

const feeAmountBigInt = feeAmount ? BigInt(feeAmount) : 10000n;
const feePerSettlement = Number(feeAmountBigInt) / 1e6;
const remainingSettlements = feeAmountBigInt > 0n
    ? Number(currentAllowance / feeAmountBigInt)
    : Infinity;

console.log(`🔍 Current Seller Allowance:`);
console.log(`   Allowance: ${Number(currentAllowance) / 1e6} ${TOKEN}`);
console.log(`   Fee per settlement: ${feePerSettlement} ${TOKEN}`);
console.log(`   Remaining settlements: ${remainingSettlements}`);

if (currentAllowance >= feeAmountBigInt) {
    console.log(`\n✅ Allowance is sufficient for at least 1 settlement`);
    console.log(`   (skip approval step if you want)`);
} else {
    console.log(`\n⚠️ Allowance insufficient — approval required!`);
}

In [ ]:
// Approve token spending for the facilitator
// Amount: 1 token = covers 100 settlements at 0.01 token fee
const APPROVAL_AMOUNT = 1_000_000n; // 1 token (6 decimals)

// Create seller wallet client for the approval transaction
const sellerWalletClient = createWalletClient({
    account: sellerAccount,
    chain: chainConfig,
    transport: http(chainConfig.rpcUrls.default.http[0]),
});

console.log(`📝 Submitting approve(${facilitatorAddr}, ${Number(APPROVAL_AMOUNT) / 1e6} ${TOKEN})...`);

const approveTxHash = await sellerWalletClient.writeContract({
    address: tokenAddress as `0x${string}`,
    abi: ERC20_ABI,
    functionName: "approve",
    args: [facilitatorAddr, APPROVAL_AMOUNT],
});

console.log(`   Tx Hash: ${approveTxHash}`);
console.log(`   Waiting for confirmation...`);

const approveReceipt = await publicClient.waitForTransactionReceipt({
    hash: approveTxHash,
});

console.log(`   ✅ Approval confirmed in block ${approveReceipt.blockNumber}`);
console.log(`   Gas used: ${approveReceipt.gasUsed}`);

// Verify new allowance
const newAllowance = await publicClient.readContract({
    address: tokenAddress as `0x${string}`,
    abi: ERC20_ABI,
    functionName: "allowance",
    args: [sellerAddr, facilitatorAddr],
});

const newRemainingSettlements = feeAmountBigInt > 0n
    ? Number(newAllowance / feeAmountBigInt)
    : Infinity;

console.log(`\n💰 Updated Allowance:`);
console.log(`   Allowance: ${Number(newAllowance) / 1e6} ${TOKEN}`);
console.log(`   Settlements available: ${newRemainingSettlements}`);

## Step 4: Create x402 Payment Payload

Now we create a standard x402 payment from the buyer to the seller using the `exact` scheme.
The buyer signs an EIP-3009 `transferWithAuthorization` that moves USDC from buyer → seller.

The facilitator fee is **not** part of this payload — it's collected separately after settlement.

In [ ]:
// Import x402 packages for payment creation
import { x402Client } from "npm:@x402/fetch@^2.17.0";
import type { PaymentRequirements, PaymentPayload, Network } from "npm:@x402/core@^2.17.0";
import { ExactEvmScheme } from "npm:@x402/evm@^2.17.0";

// Create x402 client with the standard ExactEvmScheme
const evmScheme = new ExactEvmScheme(buyerAccount);
const client = new x402Client();
client.register("eip155:*" as Network, evmScheme);
// The SDK's default spend controls only accept tokens in its own registry (USDC), so
// allowlist the token explicitly — EURC would be rejected otherwise.
client.setSpendControls({
    allowedAssets: [{ network: config.caip2Network as `${string}:${string}`, asset: tokenAddress }],
});

// Build PaymentRequirements — standard x402 exact scheme
// The seller simply receives the payment amount
const paymentRequirements: PaymentRequirements = {
    scheme: "exact",
    network: config.caip2Network,             // From network config
    amount: PAYMENT_AMOUNT,                   // What the buyer pays
    asset: tokenAddress,
    payTo: SELLER_ADDRESS as `0x${string}`,   // Seller's own address
    maxTimeoutSeconds: 3600,
    extra: {
        name: tokenName,                       // From the TOKEN selection
        version: "2"
    }
};

// Build mock 402 response (normally comes from server)
const paymentRequired = {
    x402Version: 2,
    accepts: [paymentRequirements],
    resource: {
        url: "https://example.com/resource",
        description: "exact scheme facilitator test",
        mimeType: "application/json"
    },
    extensions: {}
};

// Create payment payload — buyer signs EIP-3009 transferWithAuthorization
const paymentPayload: PaymentPayload = await client.createPaymentPayload(paymentRequired as any);

console.log("✅ Payment payload created");
console.log(`   Scheme: exact`);
console.log(`   Network: ${config.caip2Network} (${config.networkName})`);
console.log(`   From: ${paymentPayload.payload?.authorization?.from}`);
console.log(`   To (Seller): ${paymentPayload.payload?.authorization?.to}`);
console.log(`   Amount: ${paymentPayload.payload?.authorization?.value} (${Number(paymentPayload.payload?.authorization?.value || 0) / 1e6} ${TOKEN})`);
console.log(`\n💡 Payment goes directly to the seller; the facilitator fee is a`);
console.log(`   separate transferFrom, collected after settlement.`);

## Step 5: Verify Payment

Send the payment to `/verify`. The facilitator will:
1. Validate the EIP-3009 signature
2. Check buyer has sufficient USDC balance
3. **Check seller's USDC allowance for fee collection** (via `onAfterVerify` hook)

If the seller hasn't approved USDC spending, the verify response will include `invalidReason: "insufficient_fee_allowance"` with details about the required approval.

In [ ]:
// Build verify request
const verifyRequest = {
    paymentPayload: paymentPayload,
    paymentRequirements: paymentRequirements
};

console.log("🔍 Sending verification request...");

const verifyResponse = await fetch(VERIFY_URL, {
    method: "POST",
    headers: { "Content-Type": "application/json" },
    body: JSON.stringify(verifyRequest)
});

const verifyResult = await verifyResponse.json();

console.log(`\n📦 Verify Response (Status ${verifyResponse.status}):`);
console.log(JSON.stringify(verifyResult, null, 2));

if (verifyResult.isValid) {
    console.log(`\n✅ Payment is VALID!`);
    console.log(`   Payer: ${verifyResult.payer}`);
    // The facilitator's early warning: how many more settlements your current approval
    // covers. Omitted when there's no fee, or when the allowance couldn't be read.
    if (verifyResult.remainingSettlements !== undefined) {
        console.log(`   Settlements left on your approval: ${verifyResult.remainingSettlements}`);
    }
} else {
    console.log(`\n❌ Payment is INVALID!`);
    console.log(`   Reason: ${verifyResult.invalidReason}`);
    if (verifyResult.invalidReason === "insufficient_fee_allowance") {
        // /verify returns only the reason. Everything needed to fix it comes from
        // /supported (Step 1), which this notebook already read into these variables.
        console.log(`\n💡 Fix: approve more ${TOKEN}, then retry — see Step 3 above.`);
        console.log(`   Approve on ${TOKEN} contract: ${tokenAddress}`);
        console.log(`   Spender (facilitator):    ${facilitatorAddress}`);
        console.log(`   Fee per settlement:       ${Number(feeAmountBigInt) / 1e6} ${TOKEN}`);
    }
}

## Step 6: Settle Payment

Execute the settlement:
1. Facilitator calls `transferWithAuthorization` on USDC (buyer → seller)
2. **If fee is required:** Facilitator calls `transferFrom(seller, facilitator, feeAmount)` on USDC

The settle response will include a `fee` object showing whether the fee was collected.

In [ ]:
console.log(`💸 Attempting Settlement...`);
console.log(`   Network: ${paymentRequirements.network}`);
console.log(`   Buyer pays: ${Number(PAYMENT_AMOUNT) / 1e6} ${TOKEN} → Seller`);
console.log(`   Fee: ${Number(feeAmountBigInt) / 1e6} ${TOKEN} (collected after settlement)`);

// A fresh snapshot taken right here, not `balancesBefore`/`currentAllowance` (Steps 2/3) —
// those are only as fresh as whenever those cells last ran, which need not be "immediately
// before this settlement" if cells are re-run out of order. Re-running just this cell (and
// cell 19 below) repeatedly still gives Step 7 an accurate before/after comparison.
const snapshotBeforeSettle = {
    balances: {
        buyer: await getTokenBalance(buyerAccount.address),
        seller: await getTokenBalance(SELLER_ADDRESS as `0x${string}`),
        facilitator: facilitatorAddress
            ? await getTokenBalance(facilitatorAddress as `0x${string}`)
            : 0n,
    },
    allowance: await publicClient.readContract({
        address: tokenAddress as `0x${string}`,
        abi: ERC20_ABI,
        functionName: "allowance",
        args: [sellerAddr, facilitatorAddr],
    }),
};

if (USE_MAINNET) {
    console.log(`\n🚨 WARNING: REAL transaction with REAL MONEY!`);
}

const settleResponse = await fetch(SETTLE_URL, {
    method: "POST",
    headers: { "Content-Type": "application/json" },
    body: JSON.stringify(verifyRequest)
});

const settleResult = await settleResponse.json();

console.log(`\n📦 Settle Response (Status ${settleResponse.status}):`);
console.log(JSON.stringify(settleResult, null, 2));

if (settleResult.success) {
    console.log(`\n🎉 Settlement successful!`);
    console.log(`   Transaction: ${settleResult.transaction}`);
    if (settleResult.network) {
        console.log(`   Network: ${settleResult.network}`);
    }
    // Fee details
    if (settleResult.fee) {
        console.log(`\n💰 Fee Collection:`);
        console.log(`   Collected: ${settleResult.fee.collected}`);
        if (settleResult.fee.txHash) {
            console.log(`   Fee Tx: ${settleResult.fee.txHash}`);
        }
        if (settleResult.fee.error) {
            console.log(`   ⚠️ Fee Error: ${settleResult.fee.error}`);
        }
        // The receipt names the token the fee was charged in — it must be the paid token.
        const feeAsset = settleResult.extensions?.facilitatorFees?.info?.asset ?? "";
        const feeInPaidToken = feeAsset.toLowerCase().endsWith(tokenAddress.toLowerCase());
        console.log(`   Fee asset: ${feeAsset} ${feeInPaidToken ? `✅ (${TOKEN})` : "❌ not the paid token"}`);
    }
} else {
    console.log(`\n❌ Settlement failed`);
    console.log(`   Reason: ${settleResult.errorReason || "unknown"}`);
}

## Step 7: Post-Settlement Verification

Check balances and allowance after settlement to verify the complete flow:
- **Buyer** should have paid `PAYMENT_AMOUNT`
- **Seller** should have received `PAYMENT_AMOUNT - fee`
- **Facilitator** should have gained `fee`
- **Allowance** should have decreased by `fee`

In [ ]:
// Check token balances AFTER settlement
const balancesAfter = {
    buyer: await getTokenBalance(buyerAccount.address),
    seller: await getTokenBalance(SELLER_ADDRESS as `0x${string}`),
    facilitator: facilitatorAddress
        ? await getTokenBalance(facilitatorAddress as `0x${string}`)
        : 0n,
};

// Check allowance AFTER settlement
const allowanceAfter = await publicClient.readContract({
    address: tokenAddress as `0x${string}`,
    abi: ERC20_ABI,
    functionName: "allowance",
    args: [sellerAddr, facilitatorAddr],
});

console.log(`💰 ${TOKEN} Balances AFTER Settlement:`);
console.log(`   Buyer:       ${Number(balancesAfter.buyer) / 1e6} ${TOKEN}`);
console.log(`   Seller:      ${Number(balancesAfter.seller) / 1e6} ${TOKEN}`);
if (facilitatorAddress) {
    console.log(`   Facilitator: ${Number(balancesAfter.facilitator) / 1e6} ${TOKEN}`);
}

// Calculate deltas
const buyerDelta = snapshotBeforeSettle.balances.buyer - balancesAfter.buyer;
const sellerDelta = balancesAfter.seller - snapshotBeforeSettle.balances.seller;
const facilitatorDelta = balancesAfter.facilitator - snapshotBeforeSettle.balances.facilitator;
const allowanceDelta = snapshotBeforeSettle.allowance - allowanceAfter;

console.log(`\n📊 Balance Changes:`);
console.log(`   Buyer spent:          ${Number(buyerDelta) / 1e6} ${TOKEN}`);
console.log(`   Seller gained:        ${Number(sellerDelta) / 1e6} ${TOKEN}`);
console.log(`   Facilitator gained:   ${Number(facilitatorDelta) / 1e6} ${TOKEN}`);
console.log(`   Allowance decreased:  ${Number(allowanceDelta) / 1e6} ${TOKEN}`);

console.log(`\n🔍 Allowance Status:`);
console.log(`   Before: ${Number(snapshotBeforeSettle.allowance) / 1e6} ${TOKEN}`);
console.log(`   After:  ${Number(allowanceAfter) / 1e6} ${TOKEN}`);
const settlementsRemaining = feeAmountBigInt > 0n ? Number(allowanceAfter / feeAmountBigInt) : Infinity;
console.log(`   Settlements remaining: ${settlementsRemaining}`);

// Verify correctness
console.log(`\n✅ Verification:`);

// Buyer should have paid PAYMENT_AMOUNT
if (buyerDelta === BigInt(PAYMENT_AMOUNT)) {
    console.log(`   ✅ Buyer paid correct amount: ${Number(buyerDelta) / 1e6} ${TOKEN}`);
} else {
    console.log(`   ❌ Buyer payment mismatch! Expected: ${Number(PAYMENT_AMOUNT) / 1e6}, Got: ${Number(buyerDelta) / 1e6}`);
}

// Seller should have gained PAYMENT_AMOUNT - fee
const expectedSellerGain = BigInt(PAYMENT_AMOUNT) - feeAmountBigInt;
if (sellerDelta === expectedSellerGain) {
    console.log(`   ✅ Seller received correct net: ${Number(sellerDelta) / 1e6} ${TOKEN} (payment - fee)`);
} else {
    console.log(`   ❌ Seller amount mismatch! Expected: ${Number(expectedSellerGain) / 1e6}, Got: ${Number(sellerDelta) / 1e6}`);
}

// Facilitator should have gained fee
if (facilitatorDelta === feeAmountBigInt) {
    console.log(`   ✅ Facilitator collected correct fee: ${Number(facilitatorDelta) / 1e6} ${TOKEN}`);
} else {
    console.log(`   ❌ Fee mismatch! Expected: ${Number(feeAmountBigInt) / 1e6}, Got: ${Number(facilitatorDelta) / 1e6}`);
}

// Allowance should have decreased by fee
if (allowanceDelta === feeAmountBigInt) {
    console.log(`   ✅ Allowance decreased by fee amount: ${Number(allowanceDelta) / 1e6} ${TOKEN}`);
} else {
    console.log(`   ❌ Allowance mismatch! Expected decrease: ${Number(feeAmountBigInt) / 1e6}, Got: ${Number(allowanceDelta) / 1e6}`);
}

// Conservation of funds
if (sellerDelta + facilitatorDelta === buyerDelta) {
    console.log(`   ✅ Conservation of funds: Seller + Fee = Buyer spent`);
} else {
    console.log(`   ❌ Funds don't add up!`);
}

## Step 8: Block Explorer

View the transactions on the block explorer.

In [ ]:
// Block Explorer URLs (Optimism + Base)
const networkToExplorer: Record<string, string> = {
    "eip155:10": "https://optimistic.etherscan.io/tx/",
    "eip155:11155420": "https://sepolia-optimism.etherscan.io/tx/",
    "eip155:8453": "https://basescan.org/tx/",
    "eip155:84532": "https://sepolia.basescan.org/tx/",
};

const network = paymentRequirements.network;
const explorerBase = networkToExplorer[network] || "";

if (settleResult.success && settleResult.transaction) {
    const settleTxUrl = explorerBase ? `${explorerBase}${settleResult.transaction}` : settleResult.transaction;
    console.log(`🔍 Settlement Transaction:`);
    console.log(`   ${settleTxUrl}`);

    if (settleResult.fee?.txHash) {
        const feeTxUrl = explorerBase ? `${explorerBase}${settleResult.fee.txHash}` : settleResult.fee.txHash;
        console.log(`\n💰 Fee Collection Transaction:`);
        console.log(`   ${feeTxUrl}`);
    }
    
    console.log(`\n📊 Summary:`);
    console.log(`   Network: ${network} (${config.networkName})`);
    console.log(`   ${TOKEN}: ${tokenAddress}`);
    console.log(`   Buyer → Seller: ${Number(PAYMENT_AMOUNT) / 1e6} ${TOKEN}`);
    console.log(`   Seller → Facilitator: ${Number(feeAmountBigInt) / 1e6} ${TOKEN} (fee)`);
    console.log(`   Net to Seller: ${(Number(PAYMENT_AMOUNT) - Number(feeAmountBigInt)) / 1e6} ${TOKEN}`);
} else {
    console.log(`⚠️ No transaction available — settlement may have failed`);
}

## Summary

This notebook tests the `exact` scheme's fee flow end-to-end:

| Step | What happens |
|------|-------------|
| 1. `/supported` | Returns the `facilitatorFees` disclosure — fee amount & facilitator address |
| 2. Balance check | USDC balances of buyer, seller, facilitator |
| 3. Seller approval | Recurring `USDC.approve(facilitator, amount)` — small amount, enables fee collection |
| 4. Payment creation | `exact` scheme — buyer signs EIP-3009 to seller |
| 5. `/verify` | Validates signature + checks seller's allowance for the fee |
| 6. `/settle` | Executes payment on-chain, then collects the fee via `transferFrom` |
| 7. Verification | Checks all balances and the allowance changed correctly |

See `x402_facilitator/FEE_MODEL_PLAN.md` for why the fee model looks like this — in
particular Phase 5 for why the recommended approval is small and recurring rather than a
large one-time grant.

**Next:** [`x402_batch_settlement.ipynb`](./x402_batch_settlement.ipynb) walks the same
kind of test for the `batch-settlement` scheme, whose `claim`/`settle` steps charge this
same flat fee (Phase 3) but reach it through a different flow (an escrow channel instead
of a single transferWithAuthorization).